# 05 — Feature Engineering

**Project:** Predicting Corporate GHG Intensity  
**Purpose:** Build financial ratios, winsorize outliers, and run Heckman Selection Stage 1.

---


In [1]:
import sys
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from src.feature_engineering import FeatureEngineer

INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

### 1. Load Linked Panel

In [2]:
linked_df = pd.read_csv(INTERIM_DIR / "linked_panel.csv")
print("Linked Panel:", linked_df.shape)

Linked Panel: (6478, 24)


### 2. Run Feature Engineering & Heckman Selection Model
This performs winsorization, constructs corporate financial ratios, and estimates the Probit selection model using the high-emission NAICS indicator as the exclusion restriction.

In [3]:
fe = FeatureEngineer(output_dir=PROCESSED_DIR)
processed_df = fe.create_features(linked_df)
print("Engineered panel:", processed_df.shape)
print("Inverse Mills Ratio stats:")
print(processed_df["inverse_mills_ratio"].describe())

2026-09-12 21:59:58,759 - INFO - Feature engineering (full-panel) on 6478 rows …


2026-09-12 21:59:59,211 - INFO - Heckman Stage 1 (Probit, train-fold fit) — pseudo-R²: 0.4229, N = 6478, features = ['size', 'leverage', 'roa', 'high_emission_naics', 'sector_code']


2026-09-12 21:59:59,642 - INFO - Features saved: 6478 rows (3090 selected, 3388 non-selected), 52 columns → E:\Research_Projects\predicting-corporate-ghg-intensity\data\processed\processed_features.csv


Engineered panel: (6478, 52)
Inverse Mills Ratio stats:
count    6478.000000
mean        0.000619
std         0.631884
min        -1.877197
25%        -0.234758
50%        -0.030816
75%         0.316242
max         2.938420
Name: inverse_mills_ratio, dtype: float64


### Discussion & Next Steps
The feature engineering step is complete. Outliers have been winsorized at 1%/99% to limit leverage effects, and the Inverse Mills Ratio (IMR) has been calculated using the Probit model with our NAICS regulatory exclusion restriction. Next, we fit baseline linear models.